In [ ]:
!pip install -q mediapipe ultralytics onnx onnxruntime

In [ ]:
import os, json, glob, time, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from ultralytics import YOLO
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display

RANDOM_SEED = 42
CLASSES = ["bird","boar","dog","dragon","hare","horse",
           "monkey","ox","ram","rat","snake","tiger","zero"]
NUM_CLASSES = len(CLASSES)
DEVICE = [0, 1] # Use both T4 GPUs for YOLO. Use "0" for MLP (single GPU).

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
# Auto-find the dataset path in Kaggle regardless of folder structure/naming
train_dirs = glob.glob('/kaggle/input/**/train', recursive=True)

if train_dirs:
    EXTRACT_DIR = os.path.dirname(train_dirs[0])
    print(f"✅ Auto-detected dataset path at: {EXTRACT_DIR}")
    print("Contents:", os.listdir(EXTRACT_DIR))
    
    # Read stats if present
    stats_file = os.path.join(EXTRACT_DIR, 'dataset_stats.json')
    if os.path.exists(stats_file):
        with open(stats_file, 'r') as f:
            stats = json.load(f)
            print(f"Loaded dataset stats (created: {stats.get('created_at', 'N/A')})")
else:
    print("❌ ERROR: Could not find 'train' folder under /kaggle/input/. Check dataset attachment.")

In [ ]:
# Distribution Visualization
counts = {'train': [], 'val': [], 'test': []}
for split in counts.keys():
    for c in CLASSES:
        path = os.path.join(EXTRACT_DIR, split, c)
        counts[split].append(len(os.listdir(path)) if os.path.exists(path) else 0)

x = np.arange(NUM_CLASSES)
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width, counts['train'], width, label='Train')
ax.bar(x, counts['val'], width, label='Val')
ax.bar(x + width, counts['test'], width, label='Test')

ax.set_ylabel('Images')
ax.set_title('Dataset Class Distribution')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES)
ax.legend()
plt.show()

In [ ]:
# Train
model_a = YOLO('yolov8n-cls.pt')
results_a = model_a.train(
    data=EXTRACT_DIR,
    epochs=50,
    imgsz=224,
    batch=64,
    device=DEVICE,
    project='naruto_cv',
    name='approach_a_yolo_images',
    patience=10,
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3,
    weight_decay=0.0005,
    label_smoothing=0.1,
    flipud=0.0, fliplr=0.0, # Disable flips!
    pretrained=True
)

In [ ]:
# Validate
metrics_a = model_a.val()
acc1_a = metrics_a.top1
acc5_a = metrics_a.top5
print(f"Approach A - Top-1 Accuracy: {acc1_a:.4f}, Top-5: {acc5_a:.4f}")

In [ ]:
# Measure Latency
dummy_img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
# warmup
for _ in range(10): model_a(dummy_img, verbose=False)
    
start = time.time()
for _ in range(100):
    model_a(dummy_img, verbose=False)
latency_a = (time.time() - start) / 100 * 1000 # ms
print(f"Approach A Inference Latency: {latency_a:.2f} ms")

In [ ]:
# The legacy Hands namespace is not included by recent MediaPipe wheels.
# HandLandmarker is the supported replacement.
import urllib.request

HAND_LANDMARKER_MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/hand_landmarker/'
    'hand_landmarker/float16/1/hand_landmarker.task'
)
HAND_LANDMARKER_MODEL_PATH = '/kaggle/working/hand_landmarker.task'

if not os.path.exists(HAND_LANDMARKER_MODEL_PATH):
    print('Downloading the MediaPipe HandLandmarker model (one time only)...')
    tmp_model_path = HAND_LANDMARKER_MODEL_PATH + '.download'
    urllib.request.urlretrieve(HAND_LANDMARKER_MODEL_URL, tmp_model_path)
    os.replace(tmp_model_path, HAND_LANDMARKER_MODEL_PATH)

# IMAGE running mode is the direct replacement for static_image_mode=True.
hand_landmarker_options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=HAND_LANDMARKER_MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
)
hand_landmarker = vision.HandLandmarker.create_from_options(hand_landmarker_options)

def extract_normalized_landmarks(img_path):
    img = np.array(Image.open(img_path).convert('RGB'))
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
    results = hand_landmarker.detect(mp_image)
    
    if not results.hand_landmarks:
        return None
    
    # Take the first detected hand, matching the previous behaviour.
    lm_list = results.hand_landmarks[0]
    
    # 1. Extract 21x3
    coords = np.array([[lm.x, lm.y, lm.z] for lm in lm_list]) # (21, 3)
    
    # 2. Normalize (Translation: wrist to origin)
    wrist = coords[0]
    coords = coords - wrist
    
    # 3. Normalize (Scale: max absolute value to 1)
    max_val = np.max(np.abs(coords))
    if max_val > 0:
        coords = coords / max_val
        
    # Return flat 63-D vector
    return coords.flatten()

# Extract for all splits
landmark_data = {'train': [], 'val': [], 'test': []}
dropped_stats = {c: 0 for c in CLASSES}
processed_stats = {c: 0 for c in CLASSES}

for split in ['train', 'val', 'test']:
    print(f"Extracting landmarks for {split}...")
    split_dir = os.path.join(EXTRACT_DIR, split)
    
    for c_idx, c in enumerate(CLASSES):
        c_dir = os.path.join(split_dir, c)
        if not os.path.exists(c_dir): continue
            
        for f in os.listdir(c_dir):
            if not f.endswith('.png'): continue
            path = os.path.join(c_dir, f)
            
            vec = extract_normalized_landmarks(path)
            if vec is not None:
                landmark_data[split].append((vec, c_idx))
                processed_stats[c] += 1
            else:
                dropped_stats[c] += 1

print("Extraction complete!")

In [ ]:
# Landmark Quality Report
total_processed = sum(processed_stats.values())
total_dropped = sum(dropped_stats.values())
print(f"Total processed: {total_processed}, Total dropped: {total_dropped} ({(total_dropped/(total_processed+total_dropped))*100:.1f}%)")

labels = list(processed_stats.keys())
proc = [processed_stats[k] for k in labels]
drop = [dropped_stats[k] for k in labels]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(labels, proc, label='Extracted successfully', color='green')
ax.bar(labels, drop, bottom=proc, label='Dropped (No hand)', color='red')
ax.set_ylabel('Images')
ax.legend()
plt.show()

In [ ]:
# Create Landmark Grids for Approach B
GRID_DIR = '/kaggle/working/landmark_grids'
os.makedirs(GRID_DIR, exist_ok=True)

for split in ['train', 'val', 'test']:
    for c in CLASSES:
        os.makedirs(os.path.join(GRID_DIR, split, c), exist_ok=True)
        
    for i, (vec, c_idx) in enumerate(landmark_data[split]):
        # Pad 63 to 64 to reshape to 8x8
        padded = np.pad(vec, (0, 1), mode='constant')
        grid = padded.reshape((8, 8))
        
        # Scale back to 0-255 for image saving
        grid_norm = ((grid + 1.0) / 2.0 * 255.0).clip(0, 255).astype(np.uint8)
        
        img = Image.fromarray(grid_norm, mode='L')
        # Resize to 64x64 using nearest to preserve pure features, or bilinear? Bilinear requested.
        img = img.resize((64, 64), Image.Resampling.BILINEAR)
        img = img.convert('RGB') # YOLO expects RGB
        
        c = CLASSES[c_idx]
        img.save(os.path.join(GRID_DIR, split, c, f"{i}.png"))
print("Landmark grids saved for Approach B.")

In [ ]:
# Train
model_b = YOLO('yolov8n-cls.pt')
results_b = model_b.train(
    data=GRID_DIR,
    epochs=80,
    imgsz=64,
    batch=128,
    device=DEVICE,
    project='naruto_cv',
    name='approach_b_yolo_landmarks',
    patience=15,
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=5,
    label_smoothing=0.1,
    flipud=0.0, fliplr=0.0
)

In [ ]:
# Validate
metrics_b = model_b.val()
acc1_b = metrics_b.top1
acc5_b = metrics_b.top5
print(f"Approach B - Top-1 Accuracy: {acc1_b:.4f}, Top-5: {acc5_b:.4f}")

In [ ]:
# Measure Latency (YOLO forward pass only)
dummy_img_b = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
for _ in range(10): model_b(dummy_img_b, verbose=False)
    
start = time.time()
for _ in range(100):
    model_b(dummy_img_b, verbose=False)
latency_b_yolo = (time.time() - start) / 100 * 1000

# Average MediaPipe extraction time
start = time.time()
for _ in range(10):
    extract_normalized_landmarks(os.path.join(EXTRACT_DIR, 'train', CLASSES[0], os.listdir(os.path.join(EXTRACT_DIR, 'train', CLASSES[0]))[0]))
latency_mp = (time.time() - start) / 10 * 1000

latency_b = latency_b_yolo + latency_mp
print(f"Approach B Inference Latency: {latency_b:.2f} ms (MediaPipe: {latency_mp:.2f} + YOLO: {latency_b_yolo:.2f})")

In [ ]:
# Define Model
class HandSignMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, NUM_CLASSES)
        )
        
    def forward(self, x):
        return self.net(x)

mlp = HandSignMLP().cuda()

In [ ]:
# DataLoaders
def make_loader(split_data, batch_size=256, shuffle=False):
    if not split_data: return None
    X = torch.tensor(np.stack([item[0] for item in split_data]), dtype=torch.float32)
    y = torch.tensor([item[1] for item in split_data], dtype=torch.long)
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(landmark_data['train'], shuffle=True)
val_loader = make_loader(landmark_data['val'])
test_loader = make_loader(landmark_data['test'])

In [ ]:
# Train Loop
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=2)

EPOCHS = 150
best_acc_c = 0.0

train_losses, val_losses, val_accs = [], [], []

for epoch in range(EPOCHS):
    mlp.train()
    running_loss = 0.0
    for X, y in train_loader:
        X, y = X.cuda(), y.cuda()
        optimizer.zero_grad()
        out = mlp(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    scheduler.step()
    train_losses.append(running_loss / len(train_loader))
    
    # Val
    mlp.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.cuda(), y.cuda()
            out = mlp(X)
            val_loss += criterion(out, y).item()
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            
    val_acc = correct / total
    val_losses.append(val_loss / len(val_loader))
    val_accs.append(val_acc)
    
    if val_acc > best_acc_c:
        best_acc_c = val_acc
        torch.save(mlp.state_dict(), 'best_mlp.pth')
        
    if (epoch+1) % 30 == 0:
        print(f"Epoch {epoch+1:03d} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_acc:.4f}")

print(f"Approach C - Best Val Accuracy: {best_acc_c:.4f}")

In [ ]:
# Evaluate Test Set
mlp.load_state_dict(torch.load('best_mlp.pth'))
mlp.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        out = mlp(X.cuda())
        all_preds.extend(out.argmax(dim=1).cpu().numpy())
        all_labels.extend(y.numpy())

acc1_c = accuracy_score(all_labels, all_preds)
print(f"Approach C - Top-1 Accuracy (Test Set): {acc1_c:.4f}")

In [ ]:
# Measure Latency
dummy_vec = torch.randn(1, 63).cuda()
for _ in range(10): mlp(dummy_vec)
start = time.time()
for _ in range(100): mlp(dummy_vec)
latency_c_mlp = (time.time() - start) / 100 * 1000

latency_c = latency_c_mlp + latency_mp
print(f"Approach C Inference Latency: {latency_c:.2f} ms (MediaPipe: {latency_mp:.2f} + MLP: {latency_c_mlp:.2f})")

In [ ]:
# Comparison Table
import pandas as pd

data = {
    'Metric': ['Top-1 Accuracy', 'Inference Latency (ms)', 'Requires MediaPipe?'],
    'App A (YOLO Images)': [f"{acc1_a:.4f}", f"{latency_a:.2f}", "No"],
    'App B (YOLO Landmarks)': [f"{acc1_b:.4f}", f"{latency_b:.2f}", "Yes"],
    'App C (MLP Landmarks)': [f"{acc1_c:.4f}", f"{latency_c:.2f}", "Yes"]
}
df = pd.DataFrame(data)
display(df)

In [ ]:
# Select Winner Based on Accuracy
best_approach = None
best_acc = max(acc1_a, acc1_b, acc1_c)

if best_acc == acc1_a:
    best_approach = 'A'
    print("🏆 Winner is Approach A!")
    path = model_a.export(format='onnx', imgsz=224)
    shutil.copy(path, '/kaggle/working/best_model_A.onnx')
    
elif best_acc == acc1_b:
    best_approach = 'B'
    print("🏆 Winner is Approach B!")
    path = model_b.export(format='onnx', imgsz=64)
    shutil.copy(path, '/kaggle/working/best_model_B.onnx')
    
else:
    best_approach = 'C'
    print("🏆 Winner is Approach C!")
    dummy_input = torch.randn(1, 63).cuda()
    torch.onnx.export(mlp, dummy_input, '/kaggle/working/best_model_C.onnx',
                      export_params=True, opset_version=14, do_constant_folding=True,
                      input_names=['input'], output_names=['output'])

print("✅ ONNX model saved to /kaggle/working/. You can download it from the Output sidebar.")
